# PBMC3k scRNA-seq Standard Analysis (Scanpy)

**Objective**: Complete standard pipeline from raw h5ad to cell type annotation

**Steps**:
1. Environment check
2. Load and inspect AnnData
3. QC filtering
4. Normalization, HVG, PCA, clustering
5. Marker genes and cell type annotation

In [ ]:
# Import libraries
import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

sc.settings.verbosity = 2
sc.logging.print_header()
sc.settings.set_figure_params(dpi=80, facecolor='white', frameon=False)

## 1. Environment Check

In [ ]:
import sys
from pathlib import Path

print(f"Python: {sys.executable}")
print(f"Working directory: {Path.cwd()}")

packages = ['scanpy', 'anndata', 'numpy', 'pandas', 'scipy', 'matplotlib']
for pkg in packages:
    try:
        mod = __import__(pkg)
        print(f"  {pkg}: {mod.__version__}")
    except:
        print(f"  {pkg}: MISSING")

## 2. Load and Inspect Data

In [ ]:
# Load raw data
adata = sc.read_h5ad('data/raw/pbmc3k_raw.h5ad')
print(f"Raw data: {adata.n_obs} cells x {adata.n_vars} genes")
print(f"Matrix type: {type(adata.X)}")
print(f"obs columns: {list(adata.obs.columns)}")
print(f"var columns: {list(adata.var.columns)}")

## 3. QC Filtering

In [ ]:
# Calculate QC metrics
adata.obs['n_genes_by_counts'] = np.asarray(adata.X.sum(axis=1)).flatten()
adata.obs['total_counts'] = np.asarray(adata.X.sum(axis=1)).flatten()

# Mitochondrial genes
adata.var['mt'] = adata.var.index.str.startswith('MT-')
adata.obs['pct_counts_mt'] = (
    adata[:, adata.var['mt']].X.sum(axis=1).A1 / adata.obs['total_counts'] * 100
)

# Thresholds
MIN_GENES, MAX_GENES = 200, 2500
MAX_PCT_MT = 5

print(f"Cells with high MT%: {(adata.obs['pct_counts_mt'] > MAX_PCT_MT).sum()}")
print(f"Cells with <{MIN_GENES} genes: {(adata.obs['n_genes_by_counts'] < MIN_GENES).sum()}")
print(f"Cells with >{MAX_GENES} genes: {(adata.obs['n_genes_by_counts'] > MAX_GENES).sum()}")

In [ ]:
# Filter cells
adata = adata[
    (adata.obs['n_genes_by_counts'] >= MIN_GENES) &
    (adata.obs['n_genes_by_counts'] <= MAX_GENES) &
    (adata.obs['pct_counts_mt'] <= MAX_PCT_MT)
].copy()
print(f"After cell filter: {adata.n_obs} cells")

# Filter genes
sc.pp.filter_genes(adata, min_cells=3)
print(f"After gene filter: {adata.n_vars} genes")

# Save raw counts
adata.layers['counts'] = adata.X.copy()

## 4. Preprocessing & Clustering

In [ ]:
# Normalize and log-transform
sc.pp.normalize_total(adata, target_sum=10000)
sc.pp.log1p(adata)

# Highly variable genes
sc.pp.highly_variable_genes(adata, n_top_genes=2000)
print(f"HVG: {adata.var['highly_variable'].sum()}")

# Scale
sc.pp.scale(adata, max_value=10)

# PCA
sc.tl.pca(adata, n_comps=30, random_state=0)
print(f"PCA shape: {adata.obsm['X_pca'].shape}")

In [ ]:
# Neighbors, UMAP, Leiden
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=30, random_state=0)
sc.tl.umap(adata, random_state=0)
sc.tl.leiden(adata, resolution=0.5, random_state=0)
print(f"Clusters: {adata.obs['leiden'].nunique()}")
print(adata.obs['leiden'].value_counts().sort_index())

## 5. Marker Genes & Cell Type Annotation

In [ ]:
# Find marker genes
sc.tl.rank_genes_groups(adata, groupby='leiden', method='wilcoxon', random_state=0)

# Get marker table
marker_df = sc.get.rank_genes_groups_df(adata, group=None, key='rank_genes_groups')
marker_df.head(10)

In [ ]:
# Annotate based on markers
# Cluster 0: ribosomal (LDHB, RPS) - B cells
# Cluster 1: cytotoxic (NKG7, GZMA) - NK cells
# Cluster 2: MHC-II (CD74, HLA-DRA) + CD79A - B cells
# Cluster 3: monocyte (FTL, FTH1, LYZ, S100A9) - Monocytes
# Cluster 4: PPBP (platelet) - Platelets

cluster_annotation = {
    '0': 'B cells',
    '1': 'NK cells',
    '2': 'B cells',
    '3': 'Monocytes',
    '4': 'Platelets'
}

adata.obs['cell_type'] = adata.obs['leiden'].map(cluster_annotation)
print(adata.obs['cell_type'].value_counts())

In [ ]:
# UMAP by cell type
sc.pl.umap(adata, color='cell_type', title='PBMC3k Cell Types')

In [ ]:
# Save annotated data
adata.write_h5ad('data/processed/pbmc3k_annotated.h5ad')
print("Saved: data/processed/pbmc3k_annotated.h5ad")

---
## Summary

| Step | Result |
|------|--------|
| Raw data | 2700 cells x 32738 genes |
| After QC | ~1705 cells x 12360 genes |
| HVG | 2000 |
| PCA | 30 PCs |
| Leiden clusters | 5 |
| Cell types | B cells, NK cells, Monocytes, Platelets |